In [ ]:
import os
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import requests
import xml.etree.ElementTree as ET
from dotenv import load_dotenv
from pathlib import Path
import time
from entsoe import EntsoePandasClient
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully")

# Price Data Ingestion (ENTSO-E)

This is the **primary notebook** for collecting day-ahead electricity prices from ENTSO-E.

## Overview
- **Data Source:** ENTSO-E Transparency Platform via `entsoe` Python package
- **Zones:** ES (Spain), NO2 (Southern Norway), DK1 (West Denmark)
- **Period:** 2023-01-01 to present (~3 years)
- **Output:** Preprocessed wide format (date + h00-h23 columns)

> **Note:** The timeframe is deliberately restricted to the most recent three years to limit the influence of structural market changes and non-stationary dynamics (e.g., the 2022 energy crisis), ensuring more consistent and interpretable results.

## Output
Data is saved directly to `data/clean/{ZONE}_preprocessed.csv` in wide format, ready for downstream processing.

---
**Data Attribution:** Data source: ENTSO-E Transparency Platform. Licensed under CC-BY 4.0.

In [ ]:
NOTEBOOK_DIR = Path().resolve()
os.chdir(NOTEBOOK_DIR)

SECRETS_PATH = Path('../../config/secrets.env')
load_dotenv(SECRETS_PATH)

ENTSOE_TOKEN = os.getenv('ENTSOE_TOKEN')

if not ENTSOE_TOKEN:
    raise ValueError("ENTSOE_TOKEN is not in secrets.env!")

# Configuration: 3 Zones (NO4 excluded - using NO2 as representative Norwegian zone)
ZONES = {
    'ES': '10YES-REE------0',   # Spain (Solar-heavy)
    'NO2': '10YNO-2--------T',  # Norway South (Hydro, main interconnector hub)
    'DK1': '10YDK-1--------W'   # Denmark West (Wind-heavy)
}

# Data directory (Project Root)
CLEAN_DIR = Path('../../data/clean')
CLEAN_DIR.mkdir(exist_ok=True, parents=True)

# Initialize ENTSO-E client
client = EntsoePandasClient(api_key=ENTSOE_TOKEN)

print(f"Configuration loaded:")
print(f"  Zones: {list(ZONES.keys())}")
print(f"  Output: {CLEAN_DIR.resolve()}")

In [ ]:
#index of returned df is pandas._libs.tslibs.timestamps.Timestamp
#columns are names as strings "h00", "h01", "h02", ...

def fetch_entsoe_data(country_code, start, end): #start & end date are in this format as a string 'YYYYMMDD' & !!! set end date d+1 of end date target

    # Convert inputs into pandas timestamps
    start_ts = pd.Timestamp(start, tz='Europe/Brussels')
    end_ts = pd.Timestamp(end, tz='Europe/Brussels')

    if (end_ts - start_ts) > pd.Timedelta(days=190):
        raise ValueError("Time interval is to large, max. 6 months is robust!")

    # Query ENTSO-E (Day-Ahead Prices A01)
    series = client.query_day_ahead_prices(
        country_code=country_code,
        start=start_ts,
        end=end_ts
    )

    # Convert to DataFrame and naming value column
    df = series.to_frame(name="price")

    # Create columns date + time (HH:MM) 
    df["date"] = df.index.date                    # z.B. 2025-12-09
    df["time"] = df.index.strftime("%H:%M")       # z.B. "00:00", "00:15", ...

    # Pivot into date entries
    df = df.pivot_table(
        index="date",
        columns="time",
        values="price",
    )

    # Sorting and naming the dataframe
    df = df.sort_index(axis=1) # Zeiten von links nach rechts
    
    df.index = pd.to_datetime(df.index).normalize()
    df.index.name = "date"
    
    df.columns = pd.to_datetime(df.columns, format="%H:%M").time
    df.columns.name = "time"
    
    #Calculating the mean and turning dataframe into hourly values
    df_hourly = df.groupby([t.hour for t in df.columns], axis=1).mean().round(2)
    df_hourly.columns = [f"h{h:02d}" for h in df_hourly.columns]
    df_hourly = df_hourly.sort_index(axis=1)

    #Check of last row, in most calls contain only one value at 00:00
    if df_hourly.iloc[-1].isna().any():
        df_hourly = df_hourly.iloc[:-1]
    
    #Check if march is included, because of Time Change resulting in one missing value
    if (df_hourly.index.month == 3).any():
            march = df_hourly.index.month == 3
            if df_hourly.loc[march].isna().any().any():
                df_hourly.loc[march] = df_hourly.loc[march].ffill(axis=1)
    
    #Final Check for missing values and returning their position
    if df_hourly.isna().any().any():
        nan_positions = df_hourly[df_hourly.isna()].stack().index.tolist()
        print("NaN found at:")
        for date, col in nan_positions:
            print(f"  {date} → {col}")
        raise ValueError("There are NaN values in hourly data.")
    
    return df_hourly

Wrapper-Function for a longer time intervall than 6 months, which in result creates a CSV and saves it in the repository.

In [ ]:
def fetch_entsoe_range(zone, zone_code, start: str, end: str): #start & end date are in this format 'YYYYMMDD'
    start_date = pd.to_datetime(start, format="%Y%m%d")
    end_date   = pd.to_datetime(end,   format="%Y%m%d")

    dfs = []
    cur = start_date

    while cur <= end_date:
        chunk_end = min(cur + pd.DateOffset(months=6) - pd.Timedelta(days=1), end_date)

        df_chunk = fetch_entsoe_data(
            country_code=zone_code,
            start=cur.strftime("%Y%m%d"),
            end=(chunk_end + pd.Timedelta(days=1)).strftime("%Y%m%d")  # +1 Tag für die Funktion
        )

        dfs.append(df_chunk)
        cur = chunk_end + pd.Timedelta(days=1)

    full_df = pd.concat(dfs)
    full_df = full_df[~full_df.index.duplicated(keep="first")].sort_index()

    out_path = CLEAN_DIR / f"{zone}_preprocessed.csv"
    full_df.to_csv(out_path)

    print(f"Saved: {out_path}")
    
    return full_df

In [ ]:
for zone, zone_code in ZONES.items():
    fetch_entsoe_range(zone, zone_code, '20230101' , '20251127' )
    
    

## Data Quality Summary

Quick overview of the collected data to verify completeness.

In [ ]:
# Data Quality Summary
print("="*60)
print("DATA QUALITY SUMMARY")
print("="*60)

for zone in ZONES.keys():
    file_path = CLEAN_DIR / f"{zone}_preprocessed.csv"
    if file_path.exists():
        df = pd.read_csv(file_path)
        print(f"\n{zone}:")
        print(f"  Date range: {df['date'].min()} to {df['date'].max()}")
        print(f"  Total days: {len(df)}")
        print(f"  Expected days (3 years): ~1095")
        print(f"  Coverage: {len(df)/1095*100:.1f}%")
        
        # Check for any remaining NaN values
        nan_count = df.iloc[:, 1:].isna().sum().sum()
        if nan_count > 0:
            print(f"  Warning: {nan_count} NaN values found")
        else:
            print(f"  Status: Complete (no missing values)")
    else:
        print(f"\n{zone}: File not found at {file_path}")

In [ ]:
# Visualize coverage as bar chart
import matplotlib.pyplot as plt

coverage_data = []
expected_days = 1095  # ~3 years

for zone in ZONES.keys():
    file_path = CLEAN_DIR / f"{zone}_preprocessed.csv"
    if file_path.exists():
        df = pd.read_csv(file_path)
        coverage_data.append({
            'Zone': zone,
            'Days': len(df),
            'Coverage (%)': len(df) / expected_days * 100
        })

if coverage_data:
    df_cov = pd.DataFrame(coverage_data)
    
    fig, ax = plt.subplots(figsize=(8, 4))
    
    colors = ['#e74c3c', '#3498db', '#2ecc71']  # ES=red, NO2=blue, DK1=green
    bars = ax.bar(df_cov['Zone'], df_cov['Days'], color=colors, edgecolor='black', linewidth=1.2)
    
    # Add coverage labels
    for bar, pct in zip(bars, df_cov['Coverage (%)']):
        ax.annotate(f'{pct:.1f}%', 
                   xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                   ha='center', va='bottom', fontweight='bold', fontsize=12)
    
    # Add expected line
    ax.axhline(y=expected_days, color='gray', linestyle='--', linewidth=2, label=f'Expected ({expected_days} days)')
    
    ax.set_xlabel('Bidding Zone', fontweight='bold')
    ax.set_ylabel('Days Collected', fontweight='bold')
    ax.set_title('Price Data Coverage by Zone', fontweight='bold', fontsize=12)
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()